In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder

import dagshub
import mlflow
import mlflow.xgboost

pd.set_option("display.max_columns", 50)

In [ ]:
dagshub.init(
    repo_owner='lshek22',
    repo_name='walmart-recruiting-store-sales-forecasting',
    mlflow=True
)

EXP_CLEANING = "XGBoost_v2_Cleaning"
EXP_FEATURES = "XGBoost_v2_Feature_Selection"
EXP_TRAINING = "XGBoost_v2_Training_v2"

DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"  
RANDOM_STATE = 42

In [ ]:
def load_raw(data_dir):
    train = pd.read_csv(os.path.join(data_dir, "train.csv.zip"), parse_dates=["Date"])
    test = pd.read_csv(os.path.join(data_dir, "test.csv.zip"), parse_dates=["Date"])
    stores = pd.read_csv(os.path.join(data_dir, "stores.csv"))
    features = pd.read_csv(os.path.join(data_dir, "features.csv.zip"), parse_dates=["Date"])
    return train, test, stores, features


def clean_and_merge(train, test, stores, features):
    feat = features.drop(columns=["IsHoliday"])

    markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]

    def merge_one(df):
        out = df.merge(stores, on="Store", how="left")
        out = out.merge(feat, on=["Store", "Date"], how="left")
        out[markdown_cols] = out[markdown_cols].fillna(0)
        out["CPI"] = out.groupby("Store")["CPI"].transform(lambda s: s.ffill().bfill())
        out["Unemployment"] = out.groupby("Store")["Unemployment"].transform(lambda s: s.ffill().bfill())
        return out

    train_m = merge_one(train)
    test_m = merge_one(test)
    return train_m, test_m


mlflow.set_experiment(EXP_CLEANING)
with mlflow.start_run(run_name="clean_merge_v2"):
    train_raw, test_raw, stores_raw, features_raw = load_raw(DATA_DIR)

    mlflow.log_param("n_train_rows_raw", len(train_raw))
    mlflow.log_param("n_test_rows_raw", len(test_raw))
    mlflow.log_param("n_stores", stores_raw["Store"].nunique())
    mlflow.log_param("markdown_fill_strategy", "fillna(0) - genuinely absent pre Nov-2011")
    mlflow.log_param("cpi_unemployment_fill_strategy", "groupby(Store) ffill/bfill")

    train_df, test_df = clean_and_merge(train_raw, test_raw, stores_raw, features_raw)

    mlflow.log_metric("n_train_rows_clean", len(train_df))
    mlflow.log_metric("n_test_rows_clean", len(test_df))
    mlflow.log_metric("pct_missing_after_clean_train", float(train_df.isna().mean().mean()))
    mlflow.log_metric("negative_weekly_sales_count", int((train_df["Weekly_Sales"] < 0).sum()))

    train_df.to_parquet("train_clean.parquet")
    test_df.to_parquet("test_clean.parquet")
    mlflow.log_artifact("train_clean.parquet")
    mlflow.log_artifact("test_clean.parquet")

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def engineer_features(df):
    out = df.copy()

    out["Year"] = out["Date"].dt.year
    out["Month"] = out["Date"].dt.month
    out["WeekOfYear"] = out["Date"].dt.isocalendar().week.astype(int)
    out["DayOfYear"] = out["Date"].dt.dayofyear

    out["Week_sin"] = np.sin(2 * np.pi * out["WeekOfYear"] / 52)
    out["Week_cos"] = np.cos(2 * np.pi * out["WeekOfYear"] / 52)

    out["IsHoliday"] = out["IsHoliday"].astype(int)

    out["MarkDown_total"] = out[MARKDOWN_COLS].sum(axis=1)
    out["MarkDown_active_count"] = (out[MARKDOWN_COLS] > 0).sum(axis=1)

    type_map = {"A": 0, "B": 1, "C": 2}
    out["Type_enc"] = out["Type"].map(type_map)

    holiday_dates = pd.to_datetime([
        "2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08", 
        "2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06", 
        "2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29", 
        "2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27",  
    ])
    out["Days_to_holiday"] = out["Date"].apply(
        lambda d: int(np.min(np.abs((holiday_dates - d).days)))
    )

    return out


candidate_features = [
    "Store", "Dept", "Type_enc", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown_total", "MarkDown_active_count",
    "Year", "Month", "WeekOfYear", "Week_sin", "Week_cos",
    "IsHoliday", "Days_to_holiday",
]

mlflow.set_experiment(EXP_FEATURES)
with mlflow.start_run(run_name="feature_engineering_v2"):
    train_fe = engineer_features(train_df)
    test_fe = engineer_features(test_df)

    mlflow.log_param("n_candidate_features", len(candidate_features))
    mlflow.log_param("candidate_features", ", ".join(candidate_features))

    X_probe = train_fe[candidate_features]
    y_probe = train_fe["Weekly_Sales"]
    w_probe = np.where(train_fe["IsHoliday"] == 1, 5, 1)

    probe_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    probe_model.fit(X_probe, y_probe, sample_weight=w_probe)

    importances = pd.Series(probe_model.feature_importances_, index=candidate_features)
    importances = importances.sort_values(ascending=False)

    threshold = 0.005
    selected_features = importances[importances >= threshold].index.tolist()
    dropped_features = importances[importances < threshold].index.tolist()

    mlflow.log_param("selection_threshold", threshold)
    mlflow.log_param("n_selected_features", len(selected_features))
    mlflow.log_param("selected_features", ", ".join(selected_features))
    if dropped_features:
        mlflow.log_param("dropped_features", ", ".join(dropped_features))

    fig, ax = plt.subplots(figsize=(8, 6))
    importances.plot(kind="barh", ax=ax)
    ax.invert_yaxis()
    ax.set_title("XGBoost feature importance (probe model)")
    fig.tight_layout()
    fig.savefig("feature_importance_probe.png", dpi=150)
    mlflow.log_artifact("feature_importance_probe.png")
    plt.close(fig)

    train_fe.to_parquet("train_fe.parquet")
    test_fe.to_parquet("test_fe.parquet")
    mlflow.log_artifact("train_fe.parquet")
    mlflow.log_artifact("test_fe.parquet")

print("Selected features:", selected_features)
importances

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday == 1, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
N_SPLITS = 4

param_grid = {
    "max_depth": [5, 7],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [600],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
}
grid_keys = list(param_grid.keys())
grid_combos = list(itertools.product(*param_grid.values()))

X_all = train_fe[selected_features].copy()
y_all = train_fe["Weekly_Sales"].copy()
holiday_all = train_fe["IsHoliday"].copy()
dates_all = train_fe["Date"]

unique_dates = np.sort(dates_all.unique())
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

mlflow.set_experiment(EXP_TRAINING)

with mlflow.start_run(run_name="xgb_time_cv_search") as parent_run:
    mlflow.log_param("n_splits", N_SPLITS)
    mlflow.log_param("split_strategy", "TimeSeriesSplit over sorted unique weeks")
    mlflow.log_param("features_used", ", ".join(selected_features))

    results = []

    for combo in grid_combos:
        params = dict(zip(grid_keys, combo))
        fold_scores = []

        with mlflow.start_run(run_name=f"grid_{params}", nested=True):
            mlflow.log_params(params)

            for fold, (train_idx, val_idx) in enumerate(tscv.split(unique_dates)):
                train_dates = unique_dates[train_idx]
                val_dates = unique_dates[val_idx]

                tr_mask = dates_all.isin(train_dates)
                va_mask = dates_all.isin(val_dates)

                X_tr, y_tr, w_tr = X_all[tr_mask], y_all[tr_mask], np.where(holiday_all[tr_mask] == 1, 5, 1)
                X_va, y_va, h_va = X_all[va_mask], y_all[va_mask], holiday_all[va_mask]

                model = xgb.XGBRegressor(
                    **params, random_state=RANDOM_STATE, n_jobs=-1,
                    tree_method="hist",
                )
                model.fit(
                    X_tr, y_tr, sample_weight=w_tr,
                    eval_set=[(X_va, y_va)], verbose=False,
                )

                preds = model.predict(X_va)
                fold_wmae = wmae(y_va.values, preds, h_va.values)
                fold_scores.append(fold_wmae)
                mlflow.log_metric("fold_wmae", fold_wmae, step=fold)

            mean_wmae = float(np.mean(fold_scores))
            std_wmae = float(np.std(fold_scores))
            mlflow.log_metric("mean_cv_wmae", mean_wmae)
            mlflow.log_metric("std_cv_wmae", std_wmae)

        results.append({**params, "mean_cv_wmae": mean_wmae, "std_cv_wmae": std_wmae})
        print(params, "-> mean WMAE:", round(mean_wmae, 2), "+/-", round(std_wmae, 2))

    results_df = pd.DataFrame(results).sort_values("mean_cv_wmae")
    best_params = {k: results_df.iloc[0][k] for k in grid_keys}
    best_params["max_depth"] = int(best_params["max_depth"])
    best_params["n_estimators"] = int(best_params["n_estimators"])

    mlflow.log_param("best_params", str(best_params))
    mlflow.log_metric("best_mean_cv_wmae", float(results_df.iloc[0]["mean_cv_wmae"]))

    results_df.to_csv("cv_grid_results.csv", index=False)
    mlflow.log_artifact("cv_grid_results.csv")

    final_model = xgb.XGBRegressor(
        **best_params, random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )
    w_all = np.where(holiday_all == 1, 5, 1)
    final_model.fit(X_all, y_all, sample_weight=w_all, verbose=False)

    train_preds = final_model.predict(X_all)
    train_wmae = wmae(y_all.values, train_preds, holiday_all.values)
    mlflow.log_metric("final_train_wmae", train_wmae)

    mlflow.xgboost.log_model(final_model, artifact_path="model")

    fig, ax = plt.subplots(figsize=(8, 6))
    imp = pd.Series(final_model.feature_importances_, index=selected_features).sort_values()
    imp.plot(kind="barh", ax=ax)
    ax.set_title("Final XGBoost model — feature importance")
    fig.tight_layout()
    fig.savefig("final_feature_importance.png", dpi=150)
    mlflow.log_artifact("final_feature_importance.png")
    plt.close(fig)

results_df

In [ ]:
X_test = test_fe[selected_features]
test_preds = final_model.predict(X_test)
test_preds = np.clip(test_preds, 0, None) 

submission = pd.DataFrame({
    "Id": test_fe["Store"].astype(str) + "_" + test_fe["Dept"].astype(str) + "_" + test_fe["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_preds,
})
submission.to_csv("xgboost_v2_submission.csv", index=False)
submission.head()